# Zona dañada del huevo: `dano_v2`

**Por qué.** `dano_v1` marca bien roturas, grietas y hundidos, pero se entrenó solo con huevos rajados del dataset original. En la colección de daños del profesor ([adiacla/huevos](https://github.com/adiacla/huevos)) no marcaba manchas oscuras ni moho, y con el detector `v5` esos huevos ahora llegan a este modelo.

**Qué cambia.** Misma arquitectura, misma entrada (`[1,192,192,3]`) y salida (`[1,192,192,2]` = huevo, daño). Parte de los pesos de `dano_v1` y se reentrena con:
- los recortes originales (anotaciones del equipo en `dano/anotaciones/`), para no perder lo que ya sabe;
- recortes generados por [`v5/datos.py`](v5/datos.py): los 146 huevos con daño del profesor con su zona marcada (`v5/masks/`), manchas, moho y suciedad sintéticos con máscara exacta, y huevos sanos (máscara vacía).

`best.keras` se elige por la media del IoU de daño en el valid original y en el de v5. No sobrescribe nada: el run es `dano_v2`.

## 1. Setup
Monta Drive, clona el repo y busca el dataset, los recortes originales y los pesos de `dano_v1`.

In [ ]:
%pip install -q ultralytics ai-edge-litert

import json, os, re, shutil, subprocess, zipfile
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

MYDRIVE = Path('/content/drive/MyDrive')
Z1 = 'eggs_v2_parte1_train.zip'
hallados = [p for patron in (f'eggs_v2/{Z1}', f'*/{Z1}', f'*/*/{Z1}') for p in sorted(MYDRIVE.glob(patron))]
assert hallados, 'No encuentro los zips del dataset en tu Drive: ejecuta antes eggs_v3.ipynb.'
DRIVE_DIR = hallados[0].parent


def escribible(d):
    try:
        d.mkdir(parents=True, exist_ok=True)
        (d / '.prueba').write_text('ok'); (d / '.prueba').unlink()
        return True
    except OSError:
        return False


OUT_DIR = DRIVE_DIR if escribible(DRIVE_DIR / 'runs') else MYDRIVE / 'eggs_v3'
RUNS_DIR, EXPORTS_DIR = OUT_DIR / 'runs', OUT_DIR / 'exports'
RUNS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path('/content/eggs_v2')
REPO = Path('/content/eggs-detector')
WORK = Path('/content/dano_work')
RUN = 'dano_v2'

if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/dportilla219/eggs-detector.git', str(REPO)], check=True)
DET = REPO / 'modelo' / 'eggs_v2_fp32.tflite'   # mismo detector con el que se evaluó dano_v1 (para comparar)

if not (DATA_DIR / 'train').exists():
    for n in ['eggs_v2_parte1_train.zip', 'eggs_v2_parte2_train.zip', 'eggs_v2_parte3_train.zip', 'eggs_v2_parte4_valid_test.zip']:
        with zipfile.ZipFile(DRIVE_DIR / n) as zf:
            zf.extractall(DATA_DIR)
        print('OK', n)

# recortes originales: los que ya estén en Drive; si no, se generan (sección 2)
cand_seg = [d / 'dano' / 'seg' for d in (DRIVE_DIR, OUT_DIR)]
SEG_DIR = next((d for d in cand_seg if (d / 'meta.json').exists()), OUT_DIR / 'dano' / 'seg')
# pesos de dano_v1 (en Drive se llama v3_dano)
cand_init = [d / 'runs' / n / 'best.keras' for d in (DRIVE_DIR, OUT_DIR) for n in ('v3_dano', 'dano_v1')]
INIT = next((p for p in cand_init if p.exists()), None)

WORK.mkdir(exist_ok=True)
if not (WORK / 'data').exists():
    (WORK / 'data').symlink_to(DATA_DIR)
for f in (REPO / 'dano' / 'anotaciones').glob('*.txt'):
    shutil.copy(f, WORK)
os.chdir(WORK)
os.environ.update({'SEG_DIR': str(SEG_DIR), 'RUNS_DIR': str(RUNS_DIR), 'PYTHONPATH': str(REPO / 'dano'), 'TF_CPP_MIN_LOG_LEVEL': '2'})

import tensorflow as tf
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))
print('Recortes originales:', SEG_DIR, '(ya existen)' if (SEG_DIR / 'meta.json').exists() else '(se generan)')
print('Pesos de partida:', INIT or 'ImageNet (no encontré dano_v1 en Drive: se entrena desde cero, más épocas)')
print('Resultados en:', OUT_DIR)

## 2. Recortes originales
Si no están en Drive, se generan como en `eggs_dano.ipynb` (silueta con SAM 2.1 + anotaciones del equipo).

In [ ]:
D = REPO / 'dano'
if (SEG_DIR / 'meta.json').exists():
    print('Dataset ya generado en', SEG_DIR)
else:
    for split, lst in [('train', 'items_train.txt'), ('train', 'items_neg.txt'), ('valid', 'items_valid.txt'), ('valid', 'items_neg_valid.txt'),
                       ('test', 'items_test.txt'), ('test', 'items_neg_test.txt')]:
        !python {D}/eggmask.py {split} 999999 {lst} 2>&1 | grep -v Warn | tail -1
    SEG_DIR.mkdir(parents=True, exist_ok=True)
    !python {D}/build.py
for s in ['train', 'valid', 'test']:
    fs = list((SEG_DIR / s).glob('*.npz'))
    print(s, len(fs), 'recortes |', sum(f.name.endswith('_p.npz') for f in fs), 'con daño')

## 3. Recortes de v5
Colección del profesor (fijada al commit anotado), manchas sintéticas y huevos sanos. Las fotos reales de huevos sanos salen del dataset público, con las cajas de `v4` (solo las de train).

In [ ]:
PROFE_SRC = Path('/content/huevos_profe')
if not PROFE_SRC.exists():
    !git clone -q https://github.com/adiacla/huevos {PROFE_SRC}
    !git -C {PROFE_SRC} checkout -q 1b4e76826a6c1abfe0a4f20bb16f08b108569012
PUB_SRC = Path('/content/publico_src')
if not PUB_SRC.exists():
    !git clone -q --depth 1 https://github.com/dakshkathuria346-gif/Egg-Defect-Detection {PUB_SRC}

etq = next((p for p in [RUNS_DIR / 'v4_auditoria/etiquetas.json', REPO / 'resultados/v4/etiquetas.json'] if p.exists()), None)
reales = []
if etq:
    for e in json.load(open(etq)):
        if e['archivo'][:1].isdigit():
            continue
        c = 1 if e['clase'] == 'Intact' else 0
        f = PUB_SRC / ('non defective' if c == 1 else 'defective') / e['archivo']
        if f.exists():
            reales.append({'file': str(f), 'clase': c, 'cajas': e['cajas'], 'split': e['split']})
print('fotos reales con caja:', len(reales))
json.dump(reales, open('/content/reales.json', 'w'))

V5SEG = Path('/content/v5seg')
if V5SEG.exists():
    shutil.rmtree(V5SEG)
!python {REPO}/v5/datos.py --profe {PROFE_SRC}/imagenes --reales /content/reales.json --dano {REPO}/modelo/eggs_dano_v1_fp16.tflite --out {V5SEG} --n 0 0 0 --seg 3000 300

import cv2, numpy as np, matplotlib.pyplot as plt
d = np.load(V5SEG / 'seg/train/datos.npz')
fig, axes = plt.subplots(3, 10, figsize=(20, 6))
for ax, i in zip(axes.flat, range(30)):
    a = d['x'][i].copy(); m = d['y'][i][..., 1] > 127
    a[m] = (0.5 * a[m] + 0.5 * np.array([230, 40, 160])).astype(np.uint8)
    ax.imshow(a); ax.axis('off')
plt.tight_layout(); plt.show()

## 4. Entrenamiento
Parte de `dano_v1` (40 épocas, LR 3e-4) o, si no está en Drive, de ImageNet (80 épocas). Cada lote mezcla mitad recortes originales y mitad de v5. Si `runs/dano_v2` ya terminó, se salta.

In [ ]:
if (RUNS_DIR / RUN / 'last.keras').exists():
    print(RUN, 'ya está entrenado: no se repite.')
else:
    if (RUNS_DIR / RUN).exists():
        shutil.move(str(RUNS_DIR / RUN), str(RUNS_DIR / f'{RUN}_incompleto_{len(list(RUNS_DIR.glob(RUN + "_incompleto*")))}'))
    EP = 40 if INIT else 80
    !python {REPO}/dano/train_seg_v2.py {RUN} {EP} {V5SEG}/seg {INIT or 'ninguno'} 2>&1 | grep -vE "oneDNN|UserWarning|data_adapter|cuda_|computation_placer|absl|E0000|W0000|I0000"

## 5. Exportación y evaluación
Igual que `dano_v1`: `.tflite` FP32 y FP16, equivalencia con Keras y tubería completa en el test original con el detector `v2` (los mismos números que tiene `dano_v1` en `resultados/dano_v1/resumen.json`, para comparar).

In [ ]:
from IPython.display import Image, display
OUT = EXPORTS_DIR / RUN
if not (OUT / 'resumen.json').exists():
    if OUT.exists():
        shutil.rmtree(OUT)
    !python {REPO}/dano/export_eval.py {RUN} {OUT} {DET} 2>&1 | grep -vE "oneDNN|WARNING|absl|E0000|W0000|I0000|cuda_"
display(Image(str(OUT / 'ejemplos_test.jpg'), width=900))
!ls -la {OUT}

## 6. Descargar
`eggs_dano_v2_resultados.zip`: los `.tflite`, `resumen.json`, el historial del entrenamiento y los `.keras`.

In [ ]:
from google.colab import files
paquete = Path('/content/eggs_dano_v2_resultados')
if paquete.exists():
    shutil.rmtree(paquete)
shutil.copytree(OUT, paquete)
for f in ['hist.json', 'best.keras']:
    if (RUNS_DIR / RUN / f).exists():
        shutil.copy2(RUNS_DIR / RUN / f, paquete / f)
z = Path(shutil.make_archive(str(paquete), 'zip', paquete))
shutil.copy2(z, OUT_DIR / z.name)
print(f'Listo: {z.name} ({z.stat().st_size / 1e6:.1f} MB), copia en {OUT_DIR}')
files.download(str(z))